# Importy

In [5]:
# Standardowa biblioteka
import time
from pathlib import Path
from dataclasses import dataclass, asdict, replace

# Dane i wykresy
import numpy as np
import matplotlib.pyplot as plt

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

# Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.initializers import HeNormal, GlorotUniform
from tensorflow.keras.regularizers import L2
from tensorflow.keras.optimizers import SGD, Adam, AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, CSVLogger
from tensorflow.keras import losses
from tensorflow.keras import metrics as keras_metrics
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed

# Ładowanie danych i usunięcie niepotrzebnych kolumn

In [19]:
DATA_PATH = Path("../data/high_diamond_ranked_10min.csv")

TARGET = "blueWins"
DROP_COLUMNS = [
    "gameId",
    "redFirstBlood", "redKills", "redDeaths",
    "blueGoldDiff", "blueExperienceDiff", "redGoldDiff", "redExperienceDiff",
    "blueEliteMonsters", "redEliteMonsters",
    "blueCSPerMin", "redCSPerMin", "blueGoldPerMin", "redGoldPerMin",
]

def load_data(path):
    names = np.loadtxt(path, delimiter=",", max_rows=1, dtype=str)

    numbers = np.loadtxt(path, delimiter=",", skiprows=1, dtype=float)

    mask = ~np.isin(names, DROP_COLUMNS + [TARGET])
    target_mask = names == TARGET

    X = numbers[:,  mask]
    y = numbers[:, target_mask].ravel() 
    feature_names = names[mask]


    return X, y, feature_names


X, y, feature_names = load_data(DATA_PATH)


print(f"X shape: {X.shape}")
print(f"target check: {np.unique(y)}")
print(f"feature names: {feature_names}")

X shape: (9879, 25)
target check: [0. 1.]
feature names: ['blueWardsPlaced' 'blueWardsDestroyed' 'blueFirstBlood' 'blueKills'
 'blueDeaths' 'blueAssists' 'blueDragons' 'blueHeralds'
 'blueTowersDestroyed' 'blueTotalGold' 'blueAvgLevel'
 'blueTotalExperience' 'blueTotalMinionsKilled'
 'blueTotalJungleMinionsKilled' 'redWardsPlaced' 'redWardsDestroyed'
 'redAssists' 'redDragons' 'redHeralds' 'redTowersDestroyed'
 'redTotalGold' 'redAvgLevel' 'redTotalExperience' 'redTotalMinionsKilled'
 'redTotalJungleMinionsKilled']


# Konfiguracja

In [ ]:
from dataclasses import dataclass, asdict, replace

@dataclass
class ExperimentConfig:
    name: str
    hidden_layers: int =2
    units: int = 32
    dropout: float = 0.0
    l2: float = 0.0
    optimizer: str = 'adam'
    learning_rate: float = 1e-3
    momentum: float = 0.0
    weight_decay: float = 0.0
    clipnorm: float | None = None
    batch_size: int = 32
    epochs: int = 100
    early_stopping: bool = False
    reduce_lr: bool = False
    seed: int = 2026

# Funkcje pomocnicze

In [ ]:
def build_model(config, n_features):
    model = Sequential()

    model.add(Input(shape=(n_features,)))

    for i in range(config.hidden_layers):
        model.add(Dense(config.units, activation='relu',
                        kernel_initializer=HeNormal(seed=config.seed + i),
                        kernel_regularizer=L2(config.l2) if config.l2 > 0.0 else None
                        ))
        if config.dropout > 0.0:
            model.add(Dropout(config.dropout))
    
    model.add(Dense(1, activation='sigmoid'))

    return model